In [1]:
# 필수 파이썬 패키지 설치 (sudo / apt 사용 안 함)
!pip install -q sentencepiece konlpy mecab-python3 tqdm

In [2]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import logging

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# macOS에서는 나눔 폰트 경로가 다를 수 있으므로
# 폰트 이름으로 설정하고, 없으면 기본 폰트로 fallback
preferred_fonts = ["NanumBarunGothic", "Nanum Gothic", "AppleGothic"]

available_fonts = set(f.name for f in fm.fontManager.ttflist)
selected = None
for name in preferred_fonts:
    if name in available_fonts:
        selected = name
        break

if selected is not None:
    plt.rcParams["font.family"] = selected
    print(f"설정된 폰트: {selected}")
else:
    print("나눔 폰트를 찾지 못해 기본 폰트를 사용합니다.")

나눔 폰트를 찾지 못해 기본 폰트를 사용합니다.


## 01.데이터 전처리

In [4]:
import os
import re
import sentencepiece as spm
import pandas as pd

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from konlpy.tag import Mecab

from tqdm import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)

2.7.1+cu118


In [5]:
dataset_dir = "datasets"

def load_parallel_data(en_path, ko_path):
    with open(en_path, 'r', encoding='utf-8') as f:
        en_lines = [line.strip() for line in f if line.strip()]
    with open(ko_path, 'r', encoding='utf-8') as f:
        ko_lines = [line.strip() for line in f if line.strip()]
    return pd.DataFrame({'eng': en_lines, 'kor': ko_lines})

train_df = load_parallel_data(
    f"{dataset_dir}/korean-english-park.train/korean-english-park.train.en",
    f"{dataset_dir}/korean-english-park.train/korean-english-park.train.ko"
)
dev_df = load_parallel_data(
    f"{dataset_dir}/korean-english-park.dev/korean-english-park.dev.en",
    f"{dataset_dir}/korean-english-park.dev/korean-english-park.dev.ko"
)
test_df = load_parallel_data(
    f"{dataset_dir}/korean-english-park.test/korean-english-park.test.en",
    f"{dataset_dir}/korean-english-park.test/korean-english-park.test.ko"
)

# 학습 시간을 고려해 train 데이터 30,000 쌍 사용
df = train_df[:30000].reset_index(drop=True)

print(f"학습 데이터: {len(df)}, 검증 데이터: {len(dev_df)}, 테스트 데이터: {len(test_df)}")

학습 데이터: 30000, 검증 데이터: 1000, 테스트 데이터: 2000


In [6]:
df.head()

,eng,kor
0,"Much of personal computing is about ""can you t...","개인용 컴퓨터 사용의 상당 부분은 ""이것보다 뛰어날 수 있느냐?"""
1,so a mention a few weeks ago about a rechargea...,모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하...
2,"Like all optical mice, But it also doesn't nee...",그러나 이것은 또한 책상도 필요로 하지 않는다.
3,uses gyroscopic sensors to control the cursor ...,"79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분..."
4,Intelligence officials have revealed a spate o...,정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔...


## 데이터 전처리 : 정제하기

In [7]:
# 형태소 분석기 초기화 (KoNLPy Mecab)
mecab = Mecab()

def preprocess_english(sentence):
    """영어 전처리: 소문자화, 구두점 분리, 영문 외 제거"""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    return sentence.strip()

def preprocess_korean(sentence):
    """한국어 전처리: Mecab 형태소 분석, 공백 정규화"""
    sentence = sentence.strip()
    morphs = mecab.morphs(sentence)
    sentence = ' '.join(morphs)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

print("슝~")

슝~


In [8]:
tqdm.pandas()

print("영어 전처리 중...")
df["eng"] = df["eng"].progress_apply(preprocess_english)

print("한국어 형태소 분석 중... (시간이 걸릴 수 있습니다)")
df["kor"] = df["kor"].progress_apply(preprocess_korean)

df.head()

영어 전처리 중...


100%|██████████| 30000/30000 [00:00<00:00, 47792.73it/s]


한국어 형태소 분석 중... (시간이 걸릴 수 있습니다)


100%|██████████| 30000/30000 [00:03<00:00, 8408.52it/s]


,eng,kor
0,much of personal computing is about can you to...,"개인 용 컴퓨터 사용 의 상당 부분 은 "" 이것 보다 뛰어날 수 있 느냐 ? """
1,so a mention a few weeks ago about a rechargea...,모든 광 마우스 와 마찬가지 로 이 광 마우스 도 책상 위 에 놓 는 마우스 패드 ...
2,"like all optical mice , but it also doesn t ne...",그러나 이것 은 또한 책상 도 필요 로 하 지 않 는다 .
3,uses gyroscopic sensors to control the cursor ...,"79 . 95 달러 하 는 이 최첨단 무선 광 마우스 는 허공 에서 팔목 , 팔 ,..."
4,intelligence officials have revealed a spate o...,정보 관리 들 은 동남 아시아 에서 의 선박 들 에 대한 많 은 ( 테러 ) 계획 ...


,eng,kor
0,much of personal computing is about can you to...,"개인 용 컴퓨터 사용 의 상당 부분 은 "" 이것 보다 뛰어날 수 있 느냐 ? """
1,so a mention a few weeks ago about a rechargea...,모든 광 마우스 와 마찬가지 로 이 광 마우스 도 책상 위 에 놓 는 마우스 패드 ...
2,"like all optical mice , but it also doesn t ne...",그러나 이것 은 또한 책상 도 필요 로 하 지 않 는다 .
3,uses gyroscopic sensors to control the cursor ...,"79 . 95 달러 하 는 이 최첨단 무선 광 마우스 는 허공 에서 팔목 , 팔 ,..."
4,intelligence officials have revealed a spate o...,정보 관리 들 은 동남 아시아 에서 의 선박 들 에 대한 많 은 ( 테러 ) 계획 ...


## 데이터 전처리 : 토큰화 

In [9]:
df["kor"].to_csv("kor_corpus.txt", index=False, header=False, encoding="utf-8")
df["eng"].to_csv("eng_corpus.txt", index=False, header=False, encoding="utf-8")

print("파일 저장 완료: kor_corpus.txt, eng_corpus.txt")

파일 저장 완료: kor_corpus.txt, eng_corpus.txt


In [11]:
vocab_size = 10000
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

# 인코더: 한국어 (소스)
spm.SentencePieceTrainer.train(
    input="kor_corpus.txt",
    model_prefix="encoder_spm",
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id,
    character_coverage=0.9995   # 한국어 커버리지
)

# 디코더: 영어 (타겟)
spm.SentencePieceTrainer.train(
    input="eng_corpus.txt",
    model_prefix="decoder_spm",
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id,
    character_coverage=1.0      # 영어 전체 커버리지
)

print("SentencePiece 모델 학습 완료!")

SentencePiece 모델 학습 완료!


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: kor_corpus.txt
  input_format: 
  model_prefix: encoder_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  

In [12]:
encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load("encoder_spm.model")

decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load("decoder_spm.model")

True

In [13]:
df

,eng,kor
0,much of personal computing is about can you to...,"개인 용 컴퓨터 사용 의 상당 부분 은 "" 이것 보다 뛰어날 수 있 느냐 ? """
1,so a mention a few weeks ago about a rechargea...,모든 광 마우스 와 마찬가지 로 이 광 마우스 도 책상 위 에 놓 는 마우스 패드 ...
2,"like all optical mice , but it also doesn t ne...",그러나 이것 은 또한 책상 도 필요 로 하 지 않 는다 .
3,uses gyroscopic sensors to control the cursor ...,"79 . 95 달러 하 는 이 최첨단 무선 광 마우스 는 허공 에서 팔목 , 팔 ,..."
4,intelligence officials have revealed a spate o...,정보 관리 들 은 동남 아시아 에서 의 선박 들 에 대한 많 은 ( 테러 ) 계획 ...
...,...,...
29995,a majority a vast majority have voted for me a...,일각 에서 는 대 법원 이 무샤라프 의 당선 은 무효 라는 판결 을 내릴 경우 그 ...
29996,cnn an off duty sheriff s deputy shot and kill...,미국 위스콘신 주 크 랜든 에서 7 일 오전 ( 현지 시간 ) 비번 이 던 보안관 ...
29997,"the deputy , tyler peterson , also worked part...",용의자 는 타일러 피터슨 은 크랜 든 보안관 겸 비 상근 직 경찰 로 근무 해 온 ...
29998,forest county sheriff keith van cleve said pet...,케이스 반 클레브 포레스트 카운티 보안관 은 피터슨 이 20 대 였 다고 전했 다 .


In [14]:
kor_sample = df["kor"][0]
eng_sample = df["eng"][0]
print("한국어 (소스):", kor_sample)
print("영어  (타겟):", eng_sample)

한국어 (소스): 개인 용 컴퓨터 사용 의 상당 부분 은 " 이것 보다 뛰어날 수 있 느냐 ? "
영어  (타겟): much of personal computing is about can you top this ?


In [15]:
enc_token = encoder_tokenizer.encode(kor_sample)
enc_token = [encoder_tokenizer.bos_id()] + enc_token + [encoder_tokenizer.eos_id()]
print("한국어 토큰 ID:", enc_token)
print("한국어 서브워드:", [encoder_tokenizer.id_to_piece(t) for t in enc_token])

한국어 토큰 ID: [1, 1066, 558, 674, 632, 100, 333, 10, 1292, 72, 225, 12, 35, 6, 432, 117, 124, 2190, 256, 43, 19, 2428, 884, 35, 2]
한국어 서브워드: ['<s>', '▁개인', '▁용', '▁컴', '퓨', '터', '▁사용', '▁의', '▁상당', '▁부', '분', '▁은', '▁"', '▁이', '것', '▁보', '다', '▁뛰어', '날', '▁수', '▁있', '▁느냐', '▁?', '▁"', '</s>']


In [16]:
enc_decoding = encoder_tokenizer.decode(enc_token)
enc_decoding

'개인 용 컴퓨터 사용 의 상당 부분 은 " 이것 보다 뛰어날 수 있 느냐 ? "'

In [17]:
class TranslationDataset(Dataset):
    def __init__(self, data, encoder_tokenizer, decoder_tokenizer, max_len):
        self.data = data
        self.encoder_tokenizer = encoder_tokenizer
        self.decoder_tokenizer = decoder_tokenizer
        self.max_len = max_len
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['kor']   # 한국어 (소스)
        trg_text = self.data.iloc[idx]['eng']   # 영어 (타겟)

        src_ids = self.encoder_tokenizer.encode(src_text)
        trg_ids = self.decoder_tokenizer.encode(trg_text)

        src_ids = src_ids[:self.max_len]

        # Decoder의 입력에는 START_TOKEN과 END_TOKEN을 추가해줍니다. 단, 최대 길이 제한을 적용시킵니다.
        trg_input = [self.bos_id] + trg_ids[:self.max_len - 2] + [self.eos_id]
        trg_label = trg_ids[:self.max_len - 1] + [self.eos_id]

        # 길이가 짧은 경우 PAD_TOKEN을 추가해줍니다.
        src_ids = src_ids + [self.pad_id] * (self.max_len - len(src_ids))
        trg_input = trg_input + [self.pad_id] * (self.max_len - len(trg_input))
        trg_label = trg_label + [self.pad_id] * (self.max_len - len(trg_label))

        return torch.tensor(src_ids), torch.tensor(trg_input), torch.tensor(trg_label)

In [18]:
MAX_LEN = 30
BATCH_SIZE = 64

# 검증 데이터 전처리
print("검증 데이터 전처리 중...")
tqdm.pandas()
dev_df["eng"] = dev_df["eng"].apply(preprocess_english)
dev_df["kor"] = dev_df["kor"].progress_apply(preprocess_korean)

train_data = TranslationDataset(df, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)
validation_data = TranslationDataset(dev_df, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Valid batches: {len(validation_loader)}")

검증 데이터 전처리 중...


100%|██████████| 1000/1000 [00:00<00:00, 7888.46it/s]

Train batches: 469, Valid batches: 16


In [19]:
for src, trg_input, trg_label in train_loader:
    print(src.shape, trg_input.shape, trg_label.shape)
    break

torch.Size([64, 30]) torch.Size([64, 30]) torch.Size([64, 30])


## 03.모델 설계 

In [20]:
## 바다나우 

class BahdanauAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim=256):
        super().__init__()
        self.W1 = nn.Linear(enc_dim, attn_dim)
        self.W2 = nn.Linear(dec_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch_size, dec_dim)
        # encoder_outputs: (src_len, batch_size, enc_dim)
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))
        attention = self.v(energy).squeeze(2)
        return nn.functional.softmax(attention, dim=1)

In [21]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=n_layers,
                          bidirectional=True, dropout=dropout if n_layers > 1 else 0)
        self.enc_hid_dim = hidden_dim * 2
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        embedded = self.dropout(embedded)
        outputs, hidden = self.rnn(embedded)
        # hidden: (2*n_layers, batch, hidden_dim) -> concat last fwd/bwd -> project
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = torch.tanh(self.fc_hidden(hidden)).unsqueeze(0)
        return outputs, hidden

In [22]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, enc_dim, attention, dropout=0.3):
        super(Decoder, self).__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim + enc_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        # input : (batch_size,)
        # hidden : (batch_size, hidden_dim)
        # encoder_outputs : (src_len, batch_size, hidden_dim)
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))

        a = self.attention(hidden[-1], encoder_outputs)

        # H에 가중치를 부여해 attention value(Context vector) 계산
        a = a.unsqueeze(1)  # a : (batch_size, 1, src_len)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # encoder_outputs : (batch_size, src_len, hidden_dim)
        context = torch.bmm(a, encoder_outputs)  # context : (batch_size, 1, hidden_dim)
        context = context.permute(1, 0, 2)  # context : (1, batch_size, hidden_dim)

        output, hidden = self.rnn(embedded, hidden)

        # 출력층에서는 현재 hidden state와 context vector를 결합하여 예측값 생성
        output = output.squeeze(0)  # output : (batch_size, hidden_dim)
        context = context.squeeze(0)  # context : (batch_size, hidden_dim)
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, output_dim)

        return prediction, hidden, a.squeeze(1)

In [23]:
import numpy as np

class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=30, bos_id = 1, eos_id=2):
        # 학습 모드에서는 trg_len 사용, 추론 모드에서는 max_len까지 동적 생성
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        # 조기 종료를 위해 tensor가 아닌 리스트 사용
        outputs = []

        # 시각화를 위해 attention 저장
        attentions = []

        # 인코더를 통해 context 생성
        encoder_outputs, hidden = self.encoder(src)

        if trg is not None:
            for t in range(0, trg.shape[0]):
                input = trg[t]
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))

        else:
		    # inference에서는 target(정답)이 없기 때문에 sos_token을 생성해줍니다.
            input = torch.full((batch_size,), bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            for t in range(max_len):
                output, hidden, attention = self.decoder(input, hidden,  encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))
                top1 = output.argmax(1)
                input = top1

                # 조기 종료 조건
                finished |= (top1 == eos_id)
                if finished.all():
                    break

        outputs = torch.cat(outputs, dim=0)
        attentions = torch.cat(attentions, dim=0)
        return outputs, attentions

    def beam_search(self, src, max_len=30, beam_size=5, bos_id=1, eos_id=2):
        encoder_outputs, hidden = self.encoder(src)
        beams = [(0.0, hidden, [], [])]
        for t in range(max_len):
            all_candidates = []
            for score, h, seq, attn_list in beams:
                if seq and seq[-1] == eos_id:
                    all_candidates.append((score, h, seq, attn_list))
                    continue
                inp = torch.full((1,), seq[-1] if seq else bos_id, dtype=torch.long, device=self.device)
                out, h_new, attn = self.decoder(inp, h, encoder_outputs)
                log_probs = torch.nn.functional.log_softmax(out, dim=-1).squeeze(0)
                topk_scores, topk_ids = torch.topk(log_probs, min(beam_size, log_probs.size(-1)))
                for i in range(topk_ids.size(0)):
                    new_score = score + topk_scores[i].item()
                    new_id = topk_ids[i].item()
                    new_seq = seq + [new_id]
                    new_attn = attn_list + [attn.squeeze(0).cpu().numpy()]
                    all_candidates.append((new_score, h_new, new_seq, new_attn))
            all_candidates.sort(key=lambda x: x[0], reverse=True)
            beams = all_candidates[:beam_size]
            if all(b[2] and b[2][-1] == eos_id for b in beams):
                break
        best = beams[0]
        seq, attn_list = best[2], best[3]
        attentions = np.array(attn_list) if attn_list else np.zeros((1, encoder_outputs.size(0)))
        return seq, attentions

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_dim = len(encoder_tokenizer)
output_dim = len(decoder_tokenizer)
emb_dim = 256
hid_dim = 512

In [25]:
enc_hid_dim = hid_dim * 2  # bidirectional
encoder = Encoder(input_dim, emb_dim, hid_dim).to(device)
attention = BahdanauAttention(enc_hid_dim, hid_dim).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, enc_hid_dim, attention).to(device)
model = Seq2SeqAttention(encoder, decoder, device).to(device)

In [26]:
print(model)

Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(10000, 256)
    (dropout): Dropout(p=0.3, inplace=False)
    (rnn): GRU(256, 512, num_layers=2, dropout=0.3, bidirectional=True)
    (fc_hidden): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=1024, out_features=256, bias=True)
      (W2): Linear(in_features=512, out_features=256, bias=True)
      (v): Linear(in_features=256, out_features=1, bias=False)
    )
    (embedding): Embedding(10000, 256)
    (dropout): Dropout(p=0.3, inplace=False)
    (rnn): GRU(256, 512)
    (fc_out): Linear(in_features=1536, out_features=10000, bias=True)
  )
)


## 4. 훈련하기

In [27]:
#1.Optimizer & Loss

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

print("슝~")

슝~


In [28]:
#2.train_step 구현하기

def train_step(model, data_loader, optimizer, criterion, epoch):
    model.train()
    epoch_loss = 0

    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}", leave=True)

    for src, trg_input, trg_label in progress_bar:
        # 모델의 입력 순서에 맞게 transpose 변환
        src = src.permute(1, 0).to(device)
        trg_input = trg_input.permute(1, 0).to(device)
        trg_label = trg_label.permute(1, 0).to(device)
        optimizer.zero_grad()

        outputs,_ = model(src, trg_input)
        
        # (trg_len, batch_size, output_dim)을 (batch_size * trg_len, output_dim)으로 변환
        outputs = outputs.reshape(-1, outputs.shape[-1])
        trg_label = trg_label.reshape(-1)

        loss = criterion(outputs, trg_label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        optimizer.step()

        epoch_loss += loss.item()

        progress_bar.set_postfix(loss=loss.item())

    return epoch_loss / len(data_loader)

print("슝~")


슝~


In [ ]:
## (3) 훈련 시작하기


import time
start_time = time.time()

EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss = train_step(model, train_loader, optimizer, criterion, epoch)
    print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}')

print(f"Total training time: {time.time() - start_time:.2f} sec")



# import time
# start_time = time.time()

# EPOCHS = 10

# for epoch in range(EPOCHS):
#     train_loss = train_step(model, train_loader, optimizer, criterion, epoch)
#     print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}')

# print(f"Total training time: {time.time() - start_time:.2f} sec")

Epoch 1: 100%|██████████| 469/469 [01:33<00:00,  4.99it/s, loss=4.88]


Epoch 1/5, Train Loss: 5.4968


Epoch 2:  94%|█████████▎| 439/469 [01:26<00:05,  5.11it/s, loss=4.55]

In [ ]:
def eval_step(model, data_loader, optimizer, criterion):
    model.eval()
    total_loss = 0

    for src, trg_input, trg_label in data_loader:
        # 모델의 입력 순서에 맞게 transpose 변환
        src = src.permute(1, 0).to(device)
        trg_input = trg_input.permute(1, 0).to(device)
        trg_label = trg_label.permute(1, 0).to(device)

        outputs,_ = model(src, trg_input)
        
        # (trg_len, batch_size, output_dim)을 (batch_size * trg_len, output_dim)으로 변환
        outputs = outputs.reshape(-1, outputs.shape[-1])
        trg_label = trg_label.reshape(-1)

        loss = criterion(outputs, trg_label)

        total_loss += loss.item()

    return total_loss / len(data_loader)

print("슝~")

In [ ]:
%%time

EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss = train_step(model, train_loader, optimizer, criterion, epoch)
    valid_loss = eval_step(model, validation_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}')

In [ ]:
import numpy as np

def evaluate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len=30, beam_size=5):
    model.eval()
    sentence = preprocess_korean(sentence)
    src_ids = encoder_tokenizer.encode(sentence)
    src_ids = src_ids[:max_len]
    src_ids = src_ids + [0] * (max_len - len(src_ids))
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)

    with torch.no_grad():
        predicted_ids, attentions = model.beam_search(src_tensor, max_len=max_len, beam_size=beam_size)

    result = []
    for token_id in predicted_ids:
        if token_id == decoder_tokenizer.eos_id():
            break
        piece = decoder_tokenizer.id_to_piece(token_id)
        if piece:
            result.append(piece)

    attn_arr = attentions if isinstance(attentions, np.ndarray) else np.array(attentions)
    if attn_arr.ndim == 1:
        attn_arr = attn_arr[np.newaxis, :]
    elif attn_arr.ndim == 3:
        attn_arr = attn_arr.squeeze(0)
    return result, sentence, attn_arr

In [ ]:
import matplotlib.pyplot as plt

def plot_attention(attention, sentence, predicted_sentence):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.matshow(attention, cmap='viridis')

    fontdict = {'fontsize': 14}

    ax.set_xticks(range(len(sentence)))
    ax.set_xticklabels(sentence, fontdict=fontdict, rotation=90)

    ax.set_yticks(range(len(predicted_sentence)))
    ax.set_yticklabels(predicted_sentence, fontdict=fontdict)

    plt.show()

In [ ]:
def translate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len=30):
    result, sentence, attention = evaluate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len)

    full_translation = ''.join(result).replace('▁', ' ').strip()
    print('입력 (한국어): %s' % sentence)
    print('번역 결과 (영어): %s' % full_translation)

    # 어텐션 시각화: 한국어 서브워드 vs 영어 서브워드
    src_pieces = [encoder_tokenizer.id_to_piece(i) for i in encoder_tokenizer.encode(sentence)]
    attention = attention[:len(result), :len(src_pieces)]

    plot_attention(attention, src_pieces, result)

In [ ]:
translate("이것은 매우 아름다운 날이다.", model, encoder_tokenizer, decoder_tokenizer, max_len=30)

In [ ]:
translate("저는 학교에 갑니다.", model, encoder_tokenizer, decoder_tokenizer, max_len=30)

In [ ]:
translate("오늘 날씨가 정말 좋습니다.", model, encoder_tokenizer, decoder_tokenizer, max_len=30)


## 5. 테스트셋 평가

In [ ]:
# 테스트 데이터 전처리
print("테스트 데이터 전처리 중...")
tqdm.pandas()
test_df["eng"] = test_df["eng"].apply(preprocess_english)
test_df["kor"] = test_df["kor"].progress_apply(preprocess_korean)

# 테스트 데이터에서 번역 예시 확인 (정답 vs 예측)
print("\n=== 테스트셋 번역 예시 (한국어 → 영어) ===\n")
for i in range(5):
    src = test_df["kor"][i]
    ref = test_df["eng"][i]
    result, _, _ = evaluate(src, model, encoder_tokenizer, decoder_tokenizer)
    predicted = ''.join(result).replace('▁', ' ').strip()
    print(f"[예시 {i+1}]")
    print(f"  입력  (한국어): {src}")
    print(f"  정답  (영어):   {ref}")
    print(f"  번역  (예측):   {predicted}")
    print()